# Project 06 — Overdispersed Counts (Poisson vs Negative-Binomial)

**Scenario.** RNA-seq-like read counts for a gene across $N$ samples. Expression depends on a covariate $x$ on the log scale. The counts are **overdispersed** — their variance far exceeds their mean — as sequencing data almost always are.

**New skill.** *Overdispersion and count models.* A Poisson GLM forces $\operatorname{Var}(y)=\mu$. We fit **both** a Poisson and a Negative-Binomial GLM and let `az.compare` (LOO) show the NB wins. The Poisson's posterior-predictive check reveals its predictions are far too tight — the signature symptom of forced equidispersion.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240601

## Step 1 — Problem & data-generating story

Counts arise from a log-linear mean and a Negative-Binomial likelihood:

$$\log(\mu_i)=\beta_0+\beta_1 x_i,\qquad y_i\sim\text{NB}(\mu_i,\alpha),\qquad \operatorname{Var}(y_i)=\mu_i+\mu_i^2/\alpha.$$

**Assumptions made explicit:** (a) samples independent; (b) the *log-mean* is linear in $x$; (c) overdispersion is constant (single $\alpha$); (d) no excess zeros beyond what the NB implies. Truth: $\beta_0=2.2$, $\beta_1=0.8$, $\alpha=2.0$ (strong overdispersion).

In [ ]:
from data.generate_data import generate
data = generate()
x, y = data['x'], data['y']
print(f"n={data['n']}, mean={y.mean():.2f}, var={y.var():.2f}, "
      f"var/mean={y.var()/y.mean():.2f} (>>1 => overdispersed)")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(y, bins=30, color='#55A868', edgecolor='white')
ax.set(xlabel='count y', ylabel='samples', title='Observed counts — heavy right tail')
plt.tight_layout()

## Step 2 — Two models: Poisson and Negative-Binomial

Both share $\log(\mu_i)=\beta_0+\beta_1 x_i$ (log link keeps $\mu>0$). The Poisson stops there; the NB adds a dispersion parameter $\alpha$.

**Priors (weakly informative on the log scale):** $\beta_0\sim N(0,2)$, $\beta_1\sim N(0,1)$, $\alpha\sim\text{Gamma}(2,0.1)$ (mean 20, heavy-tailed so it supports both mild and strong dispersion). As $\alpha\to\infty$ the NB collapses to Poisson, so NB strictly nests it.

In [ ]:
from model import build_model, fit
m_pois = build_model(data, model='poisson')
m_nb = build_model(data, model='nb')
m_nb

## Step 3 — Prior predictive check

We simulate counts implied by the NB prior and confirm they span a plausible, heavy-tailed range rather than collapsing to 0 or exploding. Count priors are easy to set absurdly: a wide prior on $\beta_0$ implies astronomically large means after exponentiation.

In [ ]:
with m_nb:
    prior = pm.sample_prior_predictive(draws=300, random_seed=RNG)
pp = prior.prior_predictive['y'].values.ravel()
pp = pp[pp < np.percentile(pp, 99)]  # clip the extreme prior tail for the plot
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(pp, bins=40, color='#4C72B0', edgecolor='white')
ax.set(xlabel='prior-implied count', ylabel='draws',
       title='NB prior predictive (99th-pct clipped) — broad, heavy-tailed')
plt.tight_layout()

## Step 4 — Inference (NUTS) for both models

Settings: `draws=1000, tune=1000, chains=4`. Log-linear count GLMs with standardized $x$ sample easily. We fit both models and keep both idatas.

In [ ]:
idata_pois = fit(data, model='poisson', draws=1000, tune=1000, chains=4, seed=101)
idata_nb = fit(data, model='nb', draws=1000, tune=1000, chains=4, seed=101)
print('poisson divergences:', int(idata_pois.sample_stats['diverging'].sum()))
print('nb divergences     :', int(idata_nb.sample_stats['diverging'].sum()))

## Step 5 — Computational diagnostics

Both models should show $\hat R\approx1.00$, healthy ESS, and 0 divergences. (A failure to converge is *not* how the Poisson reveals its inadequacy — it converges fine to a wrong-shaped predictive; the PPC is what exposes it.)

In [ ]:
print('--- Poisson ---')
print(az.summary(idata_pois, var_names=['beta0', 'beta1']))
print('--- Negative-Binomial ---')
print(az.summary(idata_nb, var_names=['beta0', 'beta1', 'alpha']))

## Step 6 — Posterior predictive checks (where Poisson fails)

Overlay each model's posterior-predictive count distribution on the data. The Poisson predictions are **far too narrow** — it cannot reproduce the observed heavy tail because it is locked to $\operatorname{Var}=\mu$. The NB, with its free dispersion, covers the spread.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
az.plot_ppc(idata_pois, num_pp_samples=100, ax=axes[0])
axes[0].set_title('Poisson PPC — predictions too tight')
az.plot_ppc(idata_nb, num_pp_samples=100, ax=axes[1])
axes[1].set_title('Negative-Binomial PPC — covers the spread')
plt.tight_layout()

In [ ]:
# Quantify: compare observed variance to the posterior-predictive variance.
for name, idata in [('Poisson', idata_pois), ('NB', idata_nb)]:
    ppy = idata.posterior_predictive['y'].values.reshape(-1, len(y))
    pred_var = ppy.var(axis=1).mean()
    print(f'{name:>8}: observed var={y.var():.1f}, predicted var~{pred_var:.1f}')

## Step 7 — Model comparison with LOO (`az.compare`)

We rank the two models by expected log predictive density via PSIS-LOO. The NB should win decisively, with the difference in `elpd_loo` many standard errors beyond zero. This is the formal version of the visual PPC story: ignoring overdispersion costs real predictive accuracy.

In [ ]:
cmp = az.compare({'poisson': idata_pois, 'negbinom': idata_nb}, ic='loo')
print(cmp[['rank', 'elpd_loo', 'p_loo', 'elpd_diff', 'dse', 'weight']])

In [ ]:
az.plot_compare(cmp); plt.tight_layout()

**Read it:** `negbinom` is rank 0; `elpd_diff` for the Poisson is large and many `dse` away from 0, so the preference is decisive. `p_loo` (effective parameters) for the Poisson may also be inflated — a classic symptom of misspecification under LOO.

## Step 8 — Decision, recovery & communication

Confirm the NB recovers the known truth, then state the effect a collaborator cares about: the fold-change in expression per unit $x$.

In [ ]:
from shared.bayes_utils import check_recovery
for res in check_recovery(idata_nb, data['truth']):
    print(res)

In [ ]:
b1 = idata_nb.posterior['beta1'].values.ravel()
fold = np.exp(b1)
lo, hi = np.percentile(fold, [3, 97])
print(f'fold-change per +1 SD of x: median={np.median(fold):.2f}, '
      f'94% [{lo:.2f}, {hi:.2f}]')
print(f'P(beta1 > 0 | data) = {float(np.mean(b1 > 0)):.3f}')

**Conclusion (for a collaborator).** Expression rises ~2-fold per standard-deviation increase in $x$ (posterior median ~2.0, 94% interval ~1.8–2.3, comfortably covering the true 2.2), and the effect is essentially certain. Crucially we used the **Negative-Binomial**: a Poisson would have reported a falsely precise effect and badly underestimated count variability. See `summary_onepager.md`.